In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 벡터 DB : Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    (https://www.pinecone.io/ 에서 api key 생성 -> .env에 추가(PINECONE_API_KEY 등록)

# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. knowledge Base 구성을 위한 데이터 생성

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법(법률)(제21065호)(20260102).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)
document_list=loader.load_and_split(text_splitter=text_splitter)
len(document_list)

193

In [5]:
# embedding : OpenAI API text-embedding-3-large
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

In [7]:
%%time
#pinecone vector database 저장
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc= Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)
# 데이터를 처음 업로드할 때 
index_name="tax-index"
# database = PineconeVectorStore.from_documents(
#     documents = document_list,
#     embedding= embedding,
#     index_name=index_name
# )
# 업로드한 벡터db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 3.27 s
Wall time: 17.6 s


# 2. 답변 생성을 위한 Retrival

In [8]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrueved_docs = database.similarity_search(query, k=3)

In [14]:
# retrueved_docs[0].page_content
retrueved_doc = "\n\n--\n\n".join([doc.page_content for doc in retrueved_docs])

In [15]:
# query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
# retrueved_docs = database.similarity_search(query, k=3) 와 아래코드는 동일함

retruever = database.as_retriever(
    search_kwargs={"k":3}
)
retrueved_docs = retruever.invoke(query)

# 3. 답변 생성

In [ ]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4.1-namo")

In [19]:
# upstate에서 받은 달러로 llm을 사용하고 싶다면
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model="solar-pro2",
    reasoning_effort="high" # 느리지만 더 깊게 추론함(low, medium)
)

In [20]:
prompt = f"""
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
- [context]는 다음과 같아요
{retrueved_doc}
- 질문 : {query}"""

In [21]:
ai_message = llm.invoke(prompt)

In [22]:
print(ai_message.content)

연봉 5,000만원인 직장인의 소득세 계산은 다음과 같이 진행됩니다. 단, 추가적인 공제 항목(의료비, 보험료, 기부금 등)은 고려하지 않았습니다.

---

### **1. 근로소득공제 계산**
- **총급여액**: 5,000만원  
- **근로소득공제**:  
  - 4,500만원 초과 ~ 1억원 이하 구간 적용  
  - 공제액 = 1,125만원 + (5,000만원 - 4,500만원) × 5% = **1,150만원**  
- **과세표준**: 5,000만원 - 1,150만원 = **3,885만원**

---

### **2. 종합소득세 계산 (누진세율 적용)**
- **세율 구간**:  
  - 1,200만원 이하: 6%  
  - 1,200만원 초과 ~ 4,600만원 이하: 15% (초과분에 적용)  
- **계산**:  
  - 1,200만원 × 6% = **72만원**  
  - (3,885만원 - 1,200만원) × 15% = 2,685만원 × 15% = **402.75만원**  
  - **산출세액**: 72만원 + 402.75만원 = **474.75만원**

---

### **3. 근로소득세액공제 적용**
- **공제율**:  
  - 산출세액이 132만원 초과 시: 102만원 + (산출세액 - 132만원) × 20% (최대 110만원)  
- **계산**:  
  - 102만원 + (474.75만원 - 132만원) × 20% = 102만원 + 685.5만원 × 20% = **102만원 + 137.1만원 = 239.1만원**  
  - **최대 한도 110만원 적용**  
- **공제 후 세액**: 474.75만원 - 110만원 = **364.75만원**

---

### **4. 지방소득세 (주민세)**
- **계산**: 소득세액의 10%  
- **금액**: 364.75만원 × 10% = **36.475만원**

---

### **최종 납부세액**
- **국세(소득세)**: **364.75만원**  
- **지방세(주민세)**: **36